# Video decomposition test
In this workbook you can decompose video file using the previously generated (simple) Gabor library, similar to Skriabine S. et al 2026. Then visualize results.

In [ ]:
from pathlib import Path
import numpy as np
from analysis_utils import downscale_binary_video
from wavelet_utils_vSpeed import loadFilterParamDict_vS, compute_and_save_dwt_vS, makeFilterParamDict_vS, saveFilterParamDict_vS, filename_fromFilterParam

In [42]:
full_screen_coverage = [-90, 0, -45, 45] # [az_left, az_right, el_bottom, el_top] full screen position in visual degrees
visual_coverage = [-90, 0, -30, 30] # [az_left, az_right, el_bottom, el_top] screen coverage in visual degrees (seen by the animal)

screen_x = 100 # horizontal screen size in pixels for the Gabor filter generation and movie analysis
screen_t = 5 # time dimension of the screen in pixels

nx = 20 # number of Gabor filters in the horizontal direction (azimuth) (y will be generated)
        # 40 is a good number. Use 5 for testing to make it fast
        
n_thetas = 8 # number of angles to generate

size_min = 4 # minimum size in visual degrees
size_max = 20 # maximum size in visual degrees
n_sizes = 6   # number of sizes to generate

freq_min = .025 # minimum frequency in cycles per visual degree
freq_max = .15 # maximum frequency in cycles per visual degree
n_freqs = 5  # number of frequencies to generate

n_phases = 2  # number of phases to generate
phase_max = np.pi # maximum phase to generate

driftmax=4 # maximum drift speed in degrees per frame
driftnum=3 # 2n+1 drift speeds will be generated

In [43]:
#calculations
az_left, az_right, el_bottom, el_top = visual_coverage

screen_y = int(screen_x * (el_top - el_bottom) / (az_right - az_left))
ny = int(nx * (el_top - el_bottom) / (az_right - az_left))

# centers in visual degrees
xs = np.linspace(az_left, az_right, nx, endpoint=False)+(az_right - az_left) / (2*nx)
ys = np.linspace(el_bottom, el_top, ny, endpoint=False)+(el_top - el_bottom) / (2*ny)

angles= np.linspace(0, np.pi, n_thetas, endpoint=False)
sizes = np.logspace(np.log10(size_min), np.log10(size_max), n_sizes)
freqs = np.logspace(np.log10(freq_min), np.log10(freq_max), n_freqs)
phases = np.linspace(0,  phase_max, n_phases, endpoint=False)
drifts=np.linspace(-driftmax, driftmax, driftnum*2+1)

print(f"Screen size: {screen_x}x{screen_y} pixels")
print(f"Full screen coverage: {full_screen_coverage} degrees")
print(f"Visual coverage: {visual_coverage} degrees")
print(f"Center positions (x_deg): {np.round(xs, 1)} degrees")
print(f"Center positions (y_deg): {np.round(ys, 1)} degrees")
print(f"Angles (degrees): {np.round(np.rad2deg(angles), 1)}")
print(f"Sizes (degrees): {sizes}")
print(f"Frequencies (cycles/degree): {freqs}")
print(f"Phases (degrees): {np.rad2deg(phases)}")
print(f"Drifts (degrees/frame): {drifts}")

Screen size: 100x66 pixels
Full screen coverage: [-90, 0, -45, 45] degrees
Visual coverage: [-90, 0, -30, 30] degrees
Center positions (x_deg): [-87.8 -83.2 -78.8 -74.2 -69.8 -65.2 -60.8 -56.2 -51.8 -47.2 -42.8 -38.2
 -33.8 -29.2 -24.8 -20.2 -15.8 -11.2  -6.8  -2.2] degrees
Center positions (y_deg): [-27.7 -23.1 -18.5 -13.8  -9.2  -4.6  -0.    4.6   9.2  13.8  18.5  23.1
  27.7] degrees
Angles (degrees): [  0.   22.5  45.   67.5  90.  112.5 135.  157.5]
Sizes (degrees): [ 4.          5.51891865  7.61461575 10.50611122 14.49559327 20.        ]
Frequencies (cycles/degree): [0.025      0.03912711 0.06123724 0.09584147 0.15      ]
Phases (degrees): [ 0. 90.]
Drifts (degrees/frame): [-4.         -2.66666667 -1.33333333  0.          1.33333333  2.66666667
  4.        ]


In [44]:
total_n=len(sizes)*len(angles)*len(freqs)*len(drifts)*len(phases)*len(xs)*len(ys)
print(f"Total number of Gabor filters to generate: {total_n}")

Total number of Gabor filters to generate: 873600


In [45]:
# a control
gabor_step=(az_right-az_left)/nx 
print(f"Control: Gabor placement step in visual degrees (x): {gabor_step:.1f}, vs size_min: {size_min:.1f} degrees. {'OK' if (gabor_step < size_min) else 'WARNING!'}")
gabor_step=(el_top-el_bottom)/ny
print(f"Control: Gabor placement step in visual degrees (y): {gabor_step:.1f}, vs size_min: {size_min:.1f} degrees. {'OK' if (gabor_step < size_min) else 'WARNING!'}")
visual_step_x=(az_right-az_left)/screen_x
print(f"Control: Gabor resolution in visual degrees (x): {visual_step_x:.1f}, vs 1/freq_max: {1/freq_max:.1f} degrees. {'OK' if (visual_step_x < 1/freq_max/4) else 'WARNING!'}")
visual_step_y=(el_top-el_bottom)/screen_y
print(f"Control: Gabor resolution in visual degrees (y): {visual_step_y:.1f}, vs 1/freq_max: {1/freq_max:.1f} degrees. {'OK' if (visual_step_y < 1/freq_max/4) else 'WARNING!'}")


Control: Gabor placement step in visual degrees (x): 4.5, vs size_min: 4.0 degrees. WARNING!
Control: Gabor placement step in visual degrees (y): 4.6, vs size_min: 4.0 degrees. WARNING!
Control: Gabor resolution in visual degrees (x): 0.9, vs 1/freq_max: 6.7 degrees. OK
Control: Gabor resolution in visual degrees (y): 0.9, vs 1/freq_max: 6.7 degrees. OK


In [46]:

temppath = r'D:\SynologyDriveSyncedDATA\PROCESSED\Waven'

wavelet_params=makeFilterParamDict_vS(screen_x, screen_y, screen_t, visual_coverage, full_screen_coverage, xs, ys, angles, sizes, freqs, drifts, phases)

_, paramname = filename_fromFilterParam(wavelet_params)
paramspath = Path(temppath) / paramname
saveFilterParamDict_vS(wavelet_params, paramspath)
print(f"Saved Gabor filter parameters to {paramspath}")


videopath = Path(temppath) / 'zebra_s0_d420.0_fps59.94_RESAMPLED30fps.mp4'
#videopath = Path(temppath) / 'zebra_s1_d420.0_fps59.94_RESAMPLED30fps.mp4'

Saved Gabor filter parameters to D:\SynologyDriveSyncedDATA\PROCESSED\Waven\gaborLibrary_vS_20_13_8_6_5_7_2.json


In [47]:

xs, ys, angles, sizes, freqs, drifts, phases, visual_coverage, full_screen_coverage, screen_t, screen_x, screen_y = loadFilterParamDict_vS(paramspath)


In [48]:
print(f"full_screen_coverage: {full_screen_coverage}, visual_coverage: {visual_coverage}")

full_screen_coverage: [-90   0 -45  45], visual_coverage: [-90   0 -30  30]


## Downsample video

In [49]:
downsampled_video_path=downscale_binary_video(videopath, full_screen_coverage, visual_coverage, screen_x, screen_y, force=False)

Generating cropped and downsampled binary video...
Output file D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_downscaled.npy already exists. Skipping generation.


## Display downsampled video result

In [50]:
# Displaying original and downsampled video frames
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from pathlib import Path

input_video_path = Path(videopath)
#output_npy_path 

out_video = np.load(downsampled_video_path, mmap_mode="r")

cap = cv2.VideoCapture(str(input_video_path))
n_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), out_video.shape[0])

def show_frame(frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        print(f"Could not read frame {frame_idx}")
        return

    input_gray = frame[:, :, 0]
    output_frame = out_video[frame_idx].T

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(input_gray, cmap="gray")
    axes[0].set_title(f"Input frame {frame_idx}")
    axes[0].axis("off")

    axes[1].imshow(output_frame, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Output frame {frame_idx}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

interact(
    show_frame,
    frame_idx=IntSlider(
        value=0,
        min=0,
        max=n_frames - 1,
        step=1,
        description="Frame"
    )
)

interactive(children=(IntSlider(value=0, description='Frame', max=12599), Output()), _dom_classes=('widget-int…

<function __main__.show_frame(frame_idx)>

## Decomposition

In [51]:
if 'WT' in globals():
    del WT # unlink the memmapped variable as we might want to regenerate the file
    
dwt_path=compute_and_save_dwt_vS(downsampled_video_path, wavelet_params,   device='cuda', force=False)

Loaded downsampled video data from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_downscaled.npy with shape (12600, 100, 66) and dtype bool
    Torch using: cuda, GPU index: 0, GPU name: NVIDIA GeForce RTX 4070 SUPER
    n_frames: 12600
    n_wavelets: 873600
    frame_size: 6600 x 5
    feature_batch_size: 1000
    output shape: (12600, 873600) -> (12600, (20, 13, 8, 6, 5, 7, 2))
(12600, 5, 100, 66)
(12600, 33000)


Wavelet feature batches: 100%|██████████| 874/874 [08:49<00:00,  1.65it/s]


| Torch (cuda:0) tensor memory usage:
| ------------------------------------------------------------
| frames_tensor       :   793.08 MB  | shape=(12600, 33000)  dtype=torch.float16
| L_chunk             :    37.77 MB  | shape=(33000, 600)  dtype=torch.float16
| output_chunk        :    14.42 MB  | shape=(12600, 600)  dtype=torch.float16
| ------------------------------------------------------------
| TENSORS:   845.26 MB (  6.9%) | RESERVED:  1032.00 MB (  8.4%) | TOTAL GPU: 12281.50 MB  
Computed wavelet transform with shape (12600, 20, 13, 8, 6, 5, 7, 2) 
Saved wavelet transform to D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_downscaled_libvS_20_13_8_6_5_7_2dwt.npy with shape (12600, 20, 13, 8, 6, 5, 7, 2) and dtype float16


## Load and display decomposition

In [52]:
WT=np.load(dwt_path, mmap_mode='r')
print(f"Loaded DWT from {dwt_path} with shape {WT.shape}")

downsampled_video=np.load(downsampled_video_path)

Loaded DWT from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED30fps_downscaled_libvS_20_13_8_6_5_7_2dwt.npy with shape (12600, 20, 13, 8, 6, 5, 7, 2)


In [56]:
from wavelet_utils_vSpeed import makeGaborFilter_visual_vS
def get_frame(xi, yi, ai, si, fi, di, oi, params):

    xs = params['xs']
    ys = params['ys']
    angles = params['angles']
    sizes = params['sigmas']
    freqs = params['frequencies']
    drifts = params['drifts']
    phases = params['offsets']
    visual_coverage = params['visual_coverage']
    screen_x = params['screen_x']
    screen_y = params['screen_y']
    screen_t = params['screen_t']

    filt = makeGaborFilter_visual_vS(
        i_deg=xs[xi],
        j_deg=ys[yi],
        size_deg=sizes[si],
        angle=angles[ai],
        freq_deg=freqs[fi],
        drift_deg=drifts[di],
        phase=phases[oi],
        visual_coverage=visual_coverage,
        screen_x=screen_x,
        screen_y=screen_y,
        screen_t=screen_t
    )
    return filt

def get_vec(filt, visual_coverage):
    # filter shape is (t, x, y)
    az_left, az_right, el_bottom, el_top = visual_coverage
    
    def get_center(frame):
        x, y = np.indices(frame.shape)
        frame = np.abs(frame) 
        total = frame.sum()

        x_center = (x * frame).sum() / total
        y_center = (y * frame).sum() / total

        return x_center, y_center
    
    def to_xy_coords(x_center, y_center):
        x_center = np.interp(x_center, np.arange(filt.shape[1]), np.linspace(az_left, az_right, filt.shape[1]))
        y_center = np.interp(y_center, np.arange(filt.shape[2]), np.linspace(el_bottom, el_top, filt.shape[2]))
        return x_center, y_center
    
    t0=filt.shape[0]//2
    if t0==0:
        x_center, y_center = get_center(filt[0])
        x_center, y_center = to_xy_coords(x_center, y_center)
        return [x_center, x_center], [y_center, y_center]
    
    t2=t0+1
    x_center0, y_center0 = get_center(filt[t0])
    x_center2, y_center2 = get_center(filt[t2])
    x_center0, y_center0 = to_xy_coords(x_center0, y_center0)
    x_center2, y_center2 = to_xy_coords(x_center2, y_center2)

    return [x_center0+(x_center0-x_center2), x_center2], [y_center0+(y_center0-y_center2), y_center2]
    

In [57]:
# interactive visualization of the DWT and video frames
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

az_left, az_right, el_bottom, el_top = visual_coverage

param_names = ["x", "y", "angle", "size", "freq", "drift", "phase"]
param_values = [xs, ys, angles, sizes, freqs, drifts, phases]

if WT.shape[0] != downsampled_video.shape[0]:
    print(f"Warning: WT has {WT.shape[0]} frames but video has {downsampled_video.shape[0]} frames!")
n_frames = downsampled_video.shape[0]

sliders = [
    widgets.IntSlider(
        value=len(vals) // 2,
        min=0,
        max=len(vals) - 1,
        step=1,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="250px")
    )
    for name, vals in zip(param_names, param_values)
]

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=n_frames - 1,
    step=1,
    description="frame",
    continuous_update=False,
    layout=widgets.Layout(width="250px")
)

out_zoom = widgets.Output()
out_full = widgets.Output()
out_overlay = widgets.Output()
out_xy = widgets.Output()

def plot_view(*args):
    xi, yi, anglei, sizei, freqi, drifti, phasei = [s.value for s in sliders]
    ti = time_slider.value

    frame = downsampled_video[ti]
    filt = get_frame(xi, yi, anglei, sizei, freqi, drifti, phasei, wavelet_params)
    xv, yv = get_vec(filt, visual_coverage)
    dx = xv[1] - xv[0]
    dy = yv[1] - yv[0]

    transient = WT[:, xi, yi, anglei, sizei, freqi, drifti,  phasei]

    # x-y plane at current time and selected non-spatial parameters
    xy_plane = WT[ti, :, :, anglei, sizei, freqi, drifti,  phasei]

    title_params = (
        f"frame={ti}, "
        f"x={xs[xi]:.2f}, y={ys[yi]:.2f}, "
        f"angle={angles[anglei]:.2f}, "
        f"size={sizes[sizei]:.2f}, "
        f"freq={freqs[freqi]:.3f}, "
        f"drift={drifts[drifti]:.2f}, "
        f"phase={phases[phasei]:.2f}"
    )

    # --- TOP RIGHT: temporal zoom plot
    with out_zoom:
        out_zoom.clear_output(wait=True)

        z0 = max(0, ti - 12)
        z1 = min(n_frames, ti + 13)

        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(np.arange(z0, z1), transient[z0:z1], marker="o")
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(z0, z1 - 1)
        ax.set_title("WT transient — zoom ±12 frames")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        ax.set_ylim(transient.min() , transient.max() )
        plt.tight_layout()
        plt.show()

    # --- MIDDLE: full temporal plot
    with out_full:
        out_full.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(transient)
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(0, n_frames - 1)
        ax.set_title("WT transient — full")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM LEFT: current video frame with Gabor overlay
    with out_overlay:
        out_overlay.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        ax.imshow(
            frame.T,
            cmap="gray",
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        filt_plot = filt[filt.shape[0] // 2].T
        v = np.max(np.abs(filt_plot))
        if v == 0:
            v = 1

        rgba = np.zeros((*filt_plot.shape, 4), dtype=float)
        pos = filt_plot > 0
        neg = filt_plot < 0

        rgba[pos, 0] = 1.0   # red = positive
        rgba[neg, 1] = 1.0   # green = negative
        rgba[..., 3] = np.abs(filt_plot) / v * 0.7

        ax.imshow(
            rgba,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )
        
        ax.arrow(
            xv[0], yv[0],   # start
            dx, dy,         # vector
            head_width=1,
            length_includes_head=True
        )

        ax.set_title("Video + selected Gabor overlay")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM RIGHT: x-y WT plane
    with out_xy:
        out_xy.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        im = ax.imshow(
            xy_plane.T,
            cmap="seismic",
            vmin=-0.4,
            vmax=0.4,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        ax.arrow(
            xv[0], yv[0],   # start
            dx, dy,         # vector
            head_width=1,
            length_includes_head=True
        )
        
        ax.scatter(xs[xi], ys[yi], c="yellow", s=60, edgecolors="black")
        ax.set_title("WT x-y plane at selected frame")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()

    sliders[0].description = "x"
    sliders[1].description = "y"

# update on slider movement
for s in sliders + [time_slider]:
    s.observe(plot_view, names="value")

slider_box = widgets.VBox([time_slider] + sliders)

top_row = widgets.HBox([slider_box, out_zoom])
bottom_row = widgets.HBox([out_overlay, out_xy])

display(top_row, out_full, bottom_row)

plot_view()

Output()